In [2]:
import pandas as pd

df = pd.read_json("../series.jsonl", lines=True)
print(df.shape)        # (row_count, col_count)
print(df.columns.tolist())

KeyboardInterrupt: 

In [4]:
import pandas as pd

df = pd.read_json("../series.jsonl", lines=True)

print("Total entries:", len(df))
print("% null description:", round(df["description"].isna().mean() * 100), "%")

print("\nType breakdown:")
print(df["type"].value_counts())

print("\nContent rating:")
print(df["content_rating"].value_counts())

print("\nTop 30 genres:")
print(df["genres"].explode().value_counts().head(30))

print("\nTop 50 tags:")
print(df["tags"].explode().value_counts().head(50))

Total entries: 558299
% null description: 34 %

Type breakdown:
type
manga     364194
novel      61373
manhwa     59437
other      45909
manhua     24923
oel         2463
Name: count, dtype: int64

Content rating:
content_rating
safe            391329
pornographic     99157
erotica          56261
suggestive       11552
Name: count, dtype: int64

Top 30 genres:
genres
Romance          144808
Drama             98058
Hentai            92428
Comedy            84878
Adult             74931
Fantasy           71668
Yaoi              48301
Slice of Life     45625
Action            44547
Supernatural      37709
Shoujo            37643
Seinen            37186
Doujinshi         35786
Josei             34841
School Life       30271
Shounen           28141
Smut              23515
Adventure         22546
Boys Love         20838
Erotica           19978
Mystery           15591
Ecchi             15469
Shounen Ai        13241
Historical        12988
Horror            11616
Psychological     11147
Mature

In [ ]:
import pandas as pd

# Load
df = pd.read_json("../series.jsonl", lines=True)

df = df[df["state"] == "active"]

# Only manga/manhwa/manhua/oel 
df = df[df["type"].isin(["manga", "manhwa", "manhua", "oel", 'other'])]

# 3. Content rating filter — adjust if you want explicit content
df = df[df["content_rating"].isin(["safe", "suggestive"])]

# 4. Normalize genres + tags to lowercase lists (they're titlecase in the API,
#    lowercase in the dump — pick one convention now and stick to it)
df["genres"] = df["genres"].apply(
    lambda x: [genre.lower() for genre in x] if isinstance(x, list) else []
)
df["tags"] = df["tags"].apply(
    lambda x: [tag.lower() for tag in x] if isinstance(x, list) else []
)

# 5. Drop columns you'll never use
KEEP = [
    "id", "title", "native_title", "type", "status",
    "year", "rating", "description", "genres", "tags",
    "cover", "authors", "total_chapters", "source",
]
df = df[[c for c in KEEP if c in df.columns]].reset_index(drop=True)

df.to_pickle("manga_clean.pkl")
print(f"Saved {len(df)} entries")

print(f"Final dataset: {len(df)} entries")
print(f"% with description: {round(df['description'].notna().mean() * 100)}%")
print(f"% with genres: {round((df['genres'].str.len() > 0).mean() * 100)}%")
print(f"% with tags: {round((df['tags'].str.len() > 0).mean() * 100)}%")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

NameError: name 'df' is not defined

In [9]:
import pandas as pd

df = pd.read_pickle("manga_clean.pkl")

print(df.shape)
print(df.dtypes)
print(df.head(3)[["id", "title", "type", "rating", "genres", "tags"]])

# Check how many have null ratings
print(f"% null rating: {round(df['rating'].isna().mean() * 100)}%")

(177991, 14)
id                  int64
title                 str
native_title          str
type                  str
status                str
year              float64
rating            float64
description           str
genres             object
tags               object
cover              object
authors            object
total_chapters    float64
source             object
dtype: object
   id                                    title    type     rating  \
0   1                                     DICE  manhwa  70.303333   
1   2                   Lunar Legend Tsukihime   manga  81.369871   
2   3  Sha Si Nanzhu Ran Hou Cheng Wei Nümotou  manhua  76.866667   

                                              genres  \
0  [action, drama, psychological, romance, supern...   
1  [action, horror, romance, supernatural, drama,...   
2  [action, comedy, fantasy, romance, drama, mart...   

                                                tags  
0  [unrequited love, game elements, swordplay, ma...